# NTT(c) & z recovery — plotting notebook (D2)

Re-plots the recovery results from CSV so the figures can be styled freely.

**Inputs (in `02_recover_c/`):**
- `16_four_op_curves.csv` — profiling-set size vs recovery accuracy (all four operations).
- `17_ntt_upper_confusion.csv`, `17_ntt_lower_confusion.csv` — first-layer NTT(c) confusion counts.

Two figures below: (1) the four-operation curves, (2) the exact TCHES **Figure 4** 2×2
(upper curve / upper confusion / lower curve / lower confusion). Classifiers are the Table-5 MLP.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
DIR = '02_recover_c/'
df  = pd.read_csv(DIR + '16_four_op_curves.csv')
cmU = pd.read_csv(DIR + '17_ntt_upper_confusion.csv', index_col=0)
cmL = pd.read_csv(DIR + '17_ntt_lower_confusion.csv', index_col=0)
plt.rcParams.update({'font.size': 11})
df

## 1. Four-operation recovery curves
`USE_LOG_X=False` = paper linear axis; `True` = show the small-n rise to 100%.

In [ ]:
USE_LOG_X = True; XMAX = 20000
def line(ax, col, mk, lab):
    d = df.dropna(subset=[col]); ax.plot(d['n_traces'], d[col]*100, mk, label=lab)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
line(ax[0], 'ntt_c_upper_3class', '-o', 'upper c[128:256]  {-1,0,1}')
line(ax[0], 'ntt_c_lower_m1_vs_rest', '-s', 'lower c[0:128]  -1 vs {0,1}')
ax[0].axhline(100, color='grey', ls=':', lw=0.8)
ax[0].set(ylim=(98.5,100.12), xlabel='number of profiling traces', ylabel='accuracy %', title='(a) NTT(c) first-layer')
ax[0].legend(loc='lower right', fontsize=9); ax[0].grid(alpha=0.3)
for col, mk, lab in [('reduce32_z_RCoI_recall','-o','reduce32(z)'),('poly_add_z_RCoI_recall','-^','poly_add(z)'),('poly_chknorm_z_RCoI_recall','-s','poly_chknorm(z)')]:
    line(ax[1], col, mk, lab)
ax[1].set(ylim=(40,101), xlabel='number of profiling traces', ylabel='RCoI 5-class recall %', title='(b) z boundary-value')
ax[1].legend(loc='lower right', fontsize=9); ax[1].grid(alpha=0.3)
for a in ax: a.set_xscale('log') if USE_LOG_X else a.set_xlim(0, XMAX)
fig.tight_layout(); fig.savefig('four_op_curves_replot.png', dpi=150); plt.show()

## 2. TCHES Figure 4 (2×2) — first-layer NTT(c)

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(9.5, 7))
def curve(a, col, title):
    d = df.dropna(subset=[col]); a.plot(d['n_traces'], d[col]*100, '-o', color='C0')
    a.axhline(100, color='grey', ls=':', lw=0.8); a.set_xscale('log')
    a.set(xlabel='number of profiling traces', ylabel='accuracy %', ylim=(98.5,100.15), title=title); a.grid(alpha=0.3)
def cmat(a, cm, title):
    M = cm.values; labs = [s.replace('pred_','') for s in cm.columns]
    im = a.imshow(M, cmap='Greys'); fig.colorbar(im, ax=a, fraction=0.046)
    a.set_xticks(range(len(labs))); a.set_yticks(range(len(labs))); a.set_xticklabels(labs); a.set_yticklabels(labs)
    a.set(xlabel='predicted label', ylabel='true label', title=title)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            a.text(j, i, f'{M[i,j]}', ha='center', va='center', color='white' if M[i,j]>M.max()/2 else 'black', fontsize=9)
curve(ax[0,0], 'ntt_c_upper_3class', '(a) upper input coefficient')
cmat(ax[0,1], cmU, '(b) upper confusion  {-1,0,1}')
curve(ax[1,0], 'ntt_c_lower_m1_vs_rest', '(c) lower input coefficient')
cmat(ax[1,1], cmL, '(d) lower confusion  {-1} vs {0,1}')
fig.suptitle('Figure 4 (reproduction) — attack results of the first-layer NTT (D2)')
fig.tight_layout(rect=[0,0,1,0.96]); fig.savefig('ntt_fig4_replot.png', dpi=150); plt.show()